In [1]:
import pandas as pd
import numpy as np

def compare_imputation(df_raw, df_clean):
    """
    So sánh giá trị Null trước/sau và chỉ tính thống kê trên các cột Metrics.
    """
    # Danh sách Metrics cần tính thống kê
    metric_cols = [
        'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
        'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 
        'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration'
    ]
    
    # Chỉ lấy những cột thực sự tồn tại trong DataFrame
    common_metrics = [c for c in metric_cols if c in df_raw.columns and c in df_clean.columns]
    
    # 1. Tổng kết Null (áp dụng cho tất cả cột để kiểm tra coverage)
    all_cols = [c for c in df_raw.columns if c in df_clean.columns]
    summary = pd.DataFrame({
        'Null_Before': df_raw[all_cols].isnull().sum(),
        'Null_After': df_clean[all_cols].isnull().sum()
    })
    summary['Filled_Count'] = summary['Null_Before'] - summary['Null_After']
    
    print("--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---")
    print(summary[summary['Filled_Count'] > 0].to_string())
    
    # 2. So sánh Mean chỉ trên các cột Metrics
    print("\n--- SO SÁNH MEAN (CHỈ CÁC CỘT METRIC) ---")
    stats_compare = pd.DataFrame({
        'Mean_Before': df_raw[common_metrics].mean(numeric_only=True),
        'Mean_After': df_clean[common_metrics].mean(numeric_only=True)
    })
    # Thêm cột % chênh lệch để dễ đánh giá độ lệch phân phối
    stats_compare['Diff_%'] = ((stats_compare['Mean_After'] - stats_compare['Mean_Before']) / stats_compare['Mean_Before'] * 100).abs()
    print(stats_compare.to_string())
    
    # 3. Mẫu thay đổi
    print("\n--- MẪU DỮ LIỆU ĐÃ THAY ĐỔI ---")
    for col in common_metrics:
        if summary.loc[col, 'Filled_Count'] > 0:
            mask = df_raw[col].isnull() & df_clean[col].notnull()
            if mask.any():
                print(f"\nCột: {col}")
                sample = pd.concat([df_raw.loc[mask, col].head(3), df_clean.loc[mask, col].head(3)], axis=1)
                sample.columns = ['Before (NaN)', 'After (Filled)']
                print(sample)

# Load data
df_raw = pd.read_parquet("../../data/mlmart_base/v3_preprocessing.parquet")
df_clean = pd.read_parquet("../../data/mlmart_base/v3_final_cleaned.parquet")

compare_imputation(df_raw, df_clean)

--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---
                          Null_Before  Null_After  Filled_Count
capacity_kw                   1132078           0       1132078
number_of_panels              1132078           0       1132078
panel                         1132078           0       1132078
inverter                      1132078           0       1132078
optimizers                    1259007           0       1259007
site_metric                   1132078           0       1132078
weather_type_id                   188           0           188
weather_is_day                    188           0           188
shortwave_radiation               188           0           188
direct_normal_irradiance          188           0           188
diffuse_solar_radiation           188           0           188
temperature_c                     188           0           188
cloud_cover_total                 188           0           188
cloud_cover_low                   188           0           18